# Sprawdzenie paczki `expert-knowledge-rag`

Wpisz pytanie klinicysty albo tekst pacjenta, zobacz, jakie **notatki** zwraca paczka z bazy, i (opcjonalnie) poproś lokalny model o odpowiedź na tych samych notatkach.

- Dane: schemat `chips_silver` (40 oznaczonych pytań `L01..L40`, 9 tekstów pacjentów `P-V…`; dla nich widać punkty odniesienia i przy trafnych notatkach).
- Retrieval zawsze używa **bge-m3**. Wybierany model tylko odpowiada.
- Potrzebne: `DATABASE_URL`, `OLLAMA_BASE_URL`. Pierwszy raz: uruchom komórki po kolei.


In [ ]:
import html
import os
import sys
from pathlib import Path

ROOT = next(
    (
        p
        for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (p / "pyproject.toml").exists()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Uruchom notebook z katalogu repozytorium (nie znaleziono pyproject.toml).")
sys.path[:0] = [str(ROOT / "scripts" / "rag"), str(ROOT / "scripts" / "db")]

import expert_rag_lib as lib
import ipywidgets as w
from IPython.display import HTML, display
from langchain_ollama import ChatOllama

from expert_knowledge_rag import ExpertFilter, ExpertKnowledgeStore, default_embeddings

SCHEMA = lib.SILVER_SCHEMA
DSN = os.environ["DATABASE_URL"]
OLLAMA = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")

store = ExpertKnowledgeStore(DSN, schema=SCHEMA)  # read-only
embeddings = default_embeddings(base_url=OLLAMA)  # bge-m3
experts_by_vignette = lib.experts_per_vignette(DSN, SCHEMA)
examples = {i["id"]: i for i in lib.silver_examples()}

if not experts_by_vignette:
    seed = w.Button(description=f"Zbuduj {SCHEMA}", button_style="warning")
    out = w.Output()

    def _seed(_):
        with out:
            lib.seed_silver(DSN)
            print("Gotowe, uruchom tę komórkę jeszcze raz.")

    seed.on_click(_seed)
    display(HTML(f"<b>Schemat {SCHEMA} jest pusty.</b>"), seed, out)
else:
    print(f"{SCHEMA}: {len(experts_by_vignette)} winiet, {len(examples)} przykładów")

## Czat

Każda tura: **Szukaj** notatek → pełny **prompt** wysyłany do modelu → **odpowiedź** modelu, wszystko w jednej komórce; tury dopisują się pod spodem. Model nie widzi poprzednich tur (prompty ewaluacji są jednorazowe: notatki + pytanie). Pierwsze uruchomienie dużego modelu może trwać kilka minut (ładowanie); `[E1]`, `[E2]`… to cytowania.

In [ ]:
CUSTOM = "(własne)"
all_vignettes = sorted(experts_by_vignette)
all_experts = sorted({e for es in experts_by_vignette.values() for e in es})
layout = w.Layout(width="95%")
style = {"description_width": "130px"}

try:
    chat_models = lib.ollama_models(OLLAMA)
except Exception as e:  # Ollama down: the retrieval part still works
    chat_models = []
    print("Ollama niedostępna:", e)

mode = w.ToggleButtons(
    options=[("Pytanie klinicysty", "question"), ("Tekst pacjenta", "patient")],
    description="Tryb",
    style=style,
)
example = w.Dropdown(
    options=[(CUSTOM, CUSTOM)]
    + [
        (f"{i['id']}: {(i.get('patient_text') or i['question'])[:80]}", i["id"])
        for i in examples.values()
    ],
    description="Przykład",
    layout=layout,
    style=style,
)
text = w.Textarea(description="Zapytanie", rows=4, layout=layout, style=style)
llm_question = w.Text(
    value=lib.PATIENT_QUESTION,
    description="Pytanie do LLM",
    layout=layout,
    style=style,
    disabled=True,
)

# retrieval
search = w.Dropdown(options=lib.SEARCHES, description="Search", style=style)
k = w.IntSlider(value=4, min=1, max=12, description="k (notatek)", style=style)
only_vignettes = w.SelectMultiple(
    options=all_vignettes, description="Tylko winiety", rows=5, style=style
)
skip_vignettes = w.SelectMultiple(
    options=all_vignettes, description="Pomiń winiety", rows=5, style=style
)
only_experts = w.SelectMultiple(
    options=all_experts, description="Tylko eksperci", rows=5, style=style
)
merge = w.Checkbox(
    value=True,
    description="merge_shared (jednostki jednego eksperta o tym samym zdaniu w jedną notatkę)",
    indent=False,
)
group = w.Checkbox(
    value=False,
    description="group_fragments (fragmenty kilku klinicystów w jedną notatkę; ignorowane przy filtrze ekspertów)",
    indent=False,
)

# generation
ask_llm = w.Checkbox(
    value=True, description="Odpowiedź modelu (wyłączone: tylko notatki)", indent=False
)
model = w.Dropdown(
    options=chat_models,
    value="qwen3.8:27b"
    if "qwen3.8:27b" in chat_models
    else (chat_models[0] if chat_models else None),
    description="Model",
    style=style,
)
think = w.Checkbox(
    value=False, description="think (model rozumuje przed odpowiedzią)", indent=False
)
num_predict = w.IntText(
    value=lib.OPTIONS["num_predict"],
    description="num_predict",
    style=style,
)

send = w.Button(description="Wyślij", button_style="primary", icon="paper-plane")
clear = w.Button(description="Wyczyść rozmowę", icon="trash")
chat = w.Output()
status = w.HTML()
turns = [0]
last: dict = {}  # item and docs of the last turn


def _sync_mode(_=None):
    llm_question.disabled = mode.value != "patient"


def _pick_example(change):
    item = examples.get(change["new"])
    if item is None:
        return
    patient = item["suite"] == "patient"
    mode.value = "patient" if patient else "question"
    text.value = item.get("patient_text") or item["question"]
    llm_question.value = item["question"] if patient else lib.PATIENT_QUESTION
    only_vignettes.value = ()
    skip_vignettes.value = tuple(v for v in item.get("exclude", []) if v in all_vignettes)


def _current_item() -> dict:
    """The example (with its labels) while its text is untouched; otherwise a label-less item."""
    patient = mode.value == "patient"
    base = examples.get(example.value)
    if base and (base.get("patient_text") or base["question"]) == text.value:
        item = dict(base)
    else:
        item = {
            "id": "custom",
            "suite": "patient" if patient else "labelled",
            "refs": [],
            "relevant_units": [],
        }
        item["patient_text" if patient else "question"] = text.value
    if patient:
        item["question"] = llm_question.value
    item["k"] = k.value
    return item


def _filter() -> ExpertFilter:
    return ExpertFilter(
        vignettes=list(only_vignettes.value) or None,
        exclude_vignettes=list(skip_vignettes.value),
        experts=list(only_experts.value) or None,
    )


def _table(docs, relevant):
    rows = []
    for n, d in enumerate(docs, 1):
        m = d.metadata
        hit = "✓" if set(m["unit_ids"]) & relevant else ""
        rows.append(
            f"<tr><td>E{n}</td><td>{hit}</td><td>{html.escape(m['expert_id'])}</td><td>{m['vignette_id']}</td>"
            f"<td>{m['score']:.3f}</td><td>{html.escape(', '.join(m.get('kinds', [])))}</td>"
            f"<td>{html.escape(', '.join(str(x) for x in m.get('directions', []) if x))}</td>"
            f"<td style='font-size:smaller'>{html.escape(', '.join(m['unit_ids']))}</td></tr>"
        )
    head = "".join(
        f"<th>{c}</th>"
        for c in ("#", "trafna", "ekspert", "winieta", "score", "kinds", "kierunki", "unit_ids")
    )
    return f"<table><tr>{head}</tr>{''.join(rows)}</table>"


def _pre(content: str) -> str:
    return f"<pre style='white-space:pre-wrap'>{html.escape(content)}</pre>"


def _show_retrieval(item, docs):
    display(HTML(f"<b>{len(docs)} notatek</b> (search: {search.value}, k={k.value})"))
    relevant = set(item["relevant_units"])
    if relevant or item.get("covers"):
        rec = lib.retrieval_record(item, docs, experts_by_vignette)
        line = f"hit: <b>{rec.get('hit')}</b>"
        if "claim_recall" in rec:
            line += f", claim recall: <b>{rec['claim_recall']:.2f}</b>"
        display(HTML(line))
    if not docs:
        return
    display(HTML(_table(docs, relevant)))
    for n, d in enumerate(docs, 1):
        m = d.metadata
        display(
            HTML(
                f"<details {'open' if n == 1 else ''}><summary><b>[E{n}]</b> {html.escape(m['expert_id'])} · {m['vignette_id']}</summary>"
                f"{_pre(d.page_content)}</details>"
            )
        )
    if item["refs"]:
        items = "".join(f"<li>{html.escape(r)}</li>" for r in item["refs"])
        display(
            HTML(
                f"<details><summary><b>Punkty odniesienia</b> (z oznaczonych tez)</summary><ul>{items}</ul></details>"
            )
        )


def _show_answer(item, docs):
    """The full prompt first (what the model gets), then the model's answer, in place."""
    messages = lib.prompt_messages(item, docs, grouped=group.value)
    parts = "".join(
        f"<div><b>{html.escape(role)}</b>{_pre(content)}</div>" for role, content in messages
    )
    display(
        HTML(f"<details open><summary><b>Prompt wysłany do modelu</b></summary>{parts}</details>")
    )
    status.value = f"<i>{html.escape(model.value)}: generuję…</i>"
    llm = ChatOllama(
        model=model.value,
        base_url=OLLAMA,
        reasoning=think.value,
        **{**lib.OPTIONS, "num_predict": num_predict.value},
    )
    try:
        text_, stats = lib.answer(item, docs, llm, grouped=group.value)
    except Exception as e:  # surface it: a silent failure looks like a missing answer
        display(
            HTML(
                f"<b style='color:#b00'>Błąd modelu {html.escape(model.value)}:</b>{_pre(repr(e))}"
            )
        )
        return
    finally:
        status.value = ""
    c = lib.checks(text_, docs)
    body = (
        _pre(text_)
        if text_
        else "<i style='color:#b00'>Model zwrócił pustą odpowiedź "
        "(przy włączonym „think” zwiększ num_predict: rozumowanie zużywa limit tokenów).</i>"
    )
    display(
        HTML(
            f"<b>Odpowiedź · {html.escape(model.value)}</b> · {stats['seconds']} s · "
            f"{stats['output_tokens']} tokenów · {stats['tokens_per_s']} tok/s{body}"
            f"cytowane notatki: <b>{c['n_cited']}</b>, błędne numery: {c['invalid_citations'] or '-'}, "
            f"cytowani eksperci: {', '.join(c['experts_cited']) or '-'}, abstained: {c['abstained']}, słów: {c['words']}"
        )
    )


def _turn(generate: bool = True):
    """One chat turn: search with the current settings, then (optionally) the model's answer.
    Every turn is appended below the previous ones; the model does not see earlier turns."""
    with chat:
        if not text.value.strip():
            print("Wpisz zapytanie albo wybierz przykład.")
            return
        if generate and not model.value:
            print("Brak modelu (Ollama niedostępna?).")
            return
        turns[0] += 1
        item = _current_item()
        query = text.value if mode.value == "patient" else item["question"]
        display(HTML(f"<hr><h4>Tura {turns[0]} · {html.escape(item['id'])}</h4>{_pre(query)}"))
        retriever = lib.make_retriever(
            store,
            embeddings,
            patient=mode.value == "patient",
            k=k.value,
            search=search.value,
            filter=_filter(),
            merge_shared=merge.value,
            group_fragments=group.value,
        )
        docs = retriever.invoke(text.value)
        last.update(item=item, docs=docs)
        _show_retrieval(item, docs)
        if generate and docs:
            _show_answer(item, docs)
        elif generate:
            display(
                HTML("<i>Brak notatek: model nie jest wywoływany (odpowiedź: NOT_IN_SOURCES).</i>")
            )


def _on_send(_=None):
    send.disabled = True
    try:
        _turn(generate=ask_llm.value)
    finally:
        send.disabled = False


def _on_clear(_=None):
    chat.clear_output()
    turns[0] = 0


mode.observe(_sync_mode, names="value")
example.observe(_pick_example, names="value")
send.on_click(_on_send)
clear.on_click(_on_clear)
_sync_mode()
display(
    w.VBox(
        [
            mode,
            example,
            text,
            llm_question,
            w.Accordion(
                children=[
                    w.VBox([search, k, only_vignettes, skip_vignettes, only_experts, merge, group]),
                    w.VBox([ask_llm, model, think, num_predict]),
                ],
                titles=("Retrieval", "Model"),
                selected_index=1,
            ),
            w.HBox([send, clear, status]),
        ]
    ),
    chat,
)